# 🔬 Projet 6 : Comparaison des Algorithmes d'Optimisation

## Optimisation Convexe (SVM/SMO) vs Non-Convexe (MLP avec SGD & Adam)

**Dataset** : MNIST (chiffres 3 vs 8) — Classification binaire

---

Ce notebook compare :
1. **SVM (RBF)** — Optimisation convexe via l'algorithme SMO
2. **MLP + SGD (Momentum)** — Optimisation non-convexe avec descente de gradient stochastique
3. **MLP + Adam** — Optimisation non-convexe avec gradient adaptatif

## 📦 Installation des dépendances

In [ ]:
# Les dépendances sont pré-installées sur Colab
# Si besoin, décommentez les lignes suivantes :
# !pip install scikit-learn torch torchvision matplotlib pandas numpy

## 📚 Imports

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print("✅ Imports réussis !")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1️⃣ Chargement et Préparation des Données (MNIST 3 vs 8)

In [ ]:
print("📥 Téléchargement et filtrage du dataset MNIST (chiffres 3 vs 8)...")

# fetch_openml télécharge le dataset MNIST officiel
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_all, y_all = mnist.data, mnist.target.astype(int)

# Filtrage pour ne garder que les classes 3 et 8
mask = (y_all == 3) | (y_all == 8)
X_filtered = X_all[mask]
y_filtered = y_all[mask]

# Conversion des labels : 3 -> 0 et 8 -> 1 (pour la classification binaire)
y_filtered = np.where(y_filtered == 3, 0, y_filtered)
y_filtered = np.where(y_filtered == 8, 1, y_filtered)

# Échantillonnage pour accélérer l'entraînement (5000 images suffisent amplement)
np.random.seed(42)
indices = np.random.choice(len(X_filtered), 5000, replace=False)
X_sub = X_filtered[indices]
y_sub = y_filtered[indices]

# Découpage Train (80%) / Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X_sub, y_sub, test_size=0.20, random_state=42)

# Normalisation des pixels (Standardisation)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Données prêtes !")
print(f"   Train set : {X_train_scaled.shape[0]} images")
print(f"   Test set  : {X_test_scaled.shape[0]} images")
print(f"   Dimensions : {X_train_scaled.shape[1]} features (pixels)")

In [ ]:
# Visualisation de quelques exemples du dataset
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle("Exemples du dataset : Chiffres 3 (haut) et 8 (bas)", fontsize=14, fontweight='bold')

# Afficher 8 exemples de chiffre 3
idx_3 = np.where(y_train == 0)[0][:8]
for i, idx in enumerate(idx_3):
    axes[0, i].imshow(X_train[idx].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')

# Afficher 8 exemples de chiffre 8
idx_8 = np.where(y_train == 1)[0][:8]
for i, idx in enumerate(idx_8):
    axes[1, i].imshow(X_train[idx].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Dictionnaires pour stocker les résultats finaux
results_time = {}
results_acc = {}
matrices_confusion = {}

## 2️⃣ Modèle Convexe : SVM (RBF) via Algorithme SMO

Le SVM résout un problème d'optimisation **convexe** : la fonction objectif est quadratique avec des contraintes linéaires. L'algorithme SMO (Sequential Minimal Optimization) est garanti de trouver le **minimum global**.

In [ ]:
print("🔄 Entraînement du SVM (Optimisation Convexe via SMO)...")
start_time = time.time()

# On utilise un noyau RBF, la bibliothèque sklearn utilise l'algorithme SMO en arrière-plan
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_duration = time.time() - start_time

y_pred_svm = svm_model.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, y_pred_svm)

results_time['SVM (RBF/SMO)'] = svm_duration
results_acc['SVM (RBF/SMO)'] = svm_acc
matrices_confusion['SVM (RBF/SMO)'] = confusion_matrix(y_test, y_pred_svm)

print(f"\n✅ SVM entraîné en {svm_duration:.3f}s")
print(f"   Accuracy Test : {svm_acc*100:.2f}%")
print(f"   Nombre de vecteurs de support : {svm_model.n_support_}")
print(f"\n📊 Rapport de classification (SVM) :")
print(classification_report(y_test, y_pred_svm, target_names=['Chiffre 3', 'Chiffre 8']))

## 3️⃣ Modèle Non-Convexe : MLP (Réseau de Neurones en PyTorch)

Le MLP a une fonction de perte **non-convexe** à cause des couches cachées non-linéaires (ReLU). Il n'y a aucune garantie de trouver le minimum global. On compare deux optimiseurs :
- **SGD + Momentum** : Descente de gradient stochastique classique
- **Adam** : Gradient adaptatif avec estimation des moments

In [ ]:
# Configuration PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device utilisé : {device}")

# Conversion des données en Tenseurs PyTorch
X_train_t = torch.FloatTensor(X_train_scaled).to(device)
y_train_t = torch.FloatTensor(y_train).view(-1, 1).to(device)
X_test_t = torch.FloatTensor(X_test_scaled).to(device)
y_test_t = torch.FloatTensor(y_test).view(-1, 1).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

print("✅ Données converties en tenseurs PyTorch")

In [ ]:
# Définition de l'architecture du MLP (784 entrées -> 128 cachés -> 1 sortie)
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

print("📐 Architecture du MLP :")
print(MLP())

In [ ]:
# Fonction d'entraînement du MLP générique selon l'optimiseur choisi
def train_mlp(optimizer_name, opt_func, lr=0.001, epochs=40):
    model = MLP().to(device)
    criterion = nn.BCELoss()  # Binary Cross Entropy Loss pour classification binaire
    
    if optimizer_name == 'MLP - SGD (Momentum)':
        optimizer = opt_func(model.parameters(), lr=lr, momentum=0.9)
    else:
        optimizer = opt_func(model.parameters(), lr=lr)
        
    history_loss_train, history_loss_test = [], []
    history_acc_train, history_acc_test = [], []
    
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
        # Évaluation à chaque époque pour tracer les courbes
        model.eval()
        with torch.no_grad():
            train_outputs = model(X_train_t)
            test_outputs = model(X_test_t)
            
            loss_train = criterion(train_outputs, y_train_t).item()
            loss_test = criterion(test_outputs, y_test_t).item()
            
            acc_train = accuracy_score(y_train, (train_outputs.cpu().numpy() > 0.5).astype(int))
            acc_test = accuracy_score(y_test, (test_outputs.cpu().numpy() > 0.5).astype(int))
            
            history_loss_train.append(loss_train)
            history_loss_test.append(loss_test)
            history_acc_train.append(acc_train)
            history_acc_test.append(acc_test)
        
        # Affichage de la progression
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"   Epoch {epoch+1:>3}/{epochs} | Loss: {loss_train:.4f} | Acc Train: {acc_train*100:.1f}% | Acc Test: {acc_test*100:.1f}%")
            
    duration = time.time() - start_time
    
    # Prédictions finales sur le Test Set
    final_preds = (model(X_test_t).cpu().numpy() > 0.5).astype(int)
    
    return duration, acc_test, final_preds, history_loss_train, history_loss_test, history_acc_train, history_acc_test

In [ ]:
# Entraînement avec SGD + Momentum
print("🔄 Entraînement du MLP avec SGD (Momentum)...")
sgd_res = train_mlp('MLP - SGD (Momentum)', optim.SGD, lr=0.01)
results_time['MLP - SGD (Momentum)'] = sgd_res[0]
results_acc['MLP - SGD (Momentum)'] = sgd_res[1]
matrices_confusion['MLP - SGD (Momentum)'] = confusion_matrix(y_test, sgd_res[2])
print(f"\n✅ SGD terminé en {sgd_res[0]:.3f}s | Accuracy finale : {sgd_res[1]*100:.2f}%")

In [ ]:
# Entraînement avec Adam
print("🔄 Entraînement du MLP avec Adam (Gradient Adaptatif)...")
adam_res = train_mlp('MLP - Adam', optim.Adam, lr=0.001)
results_time['MLP - Adam'] = adam_res[0]
results_acc['MLP - Adam'] = adam_res[1]
matrices_confusion['MLP - Adam'] = confusion_matrix(y_test, adam_res[2])
print(f"\n✅ Adam terminé en {adam_res[0]:.3f}s | Accuracy finale : {adam_res[1]*100:.2f}%")

## 4️⃣ Visualisation des Résultats

### 📈 Courbes d'apprentissage : Loss et Accuracy (SGD vs Adam)

In [ ]:
epochs_range = range(1, 41)
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Comparaison SGD (Momentum) vs Adam - Entraînement du MLP", fontsize=14, fontweight='bold', y=1.02)

# Subplot 1: Loss
ax[0].plot(epochs_range, sgd_res[3], 'r-', label='SGD - Train', linewidth=2)
ax[0].plot(epochs_range, sgd_res[4], 'r--', label='SGD - Test', linewidth=2)
ax[0].plot(epochs_range, adam_res[3], 'b-', label='Adam - Train', linewidth=2)
ax[0].plot(epochs_range, adam_res[4], 'b--', label='Adam - Test', linewidth=2)
ax[0].set_title("Évolution de la Loss au fil des époques", fontsize=12)
ax[0].set_xlabel("Époque")
ax[0].set_ylabel("Loss (Binary Cross-Entropy)")
ax[0].grid(True, alpha=0.3)
ax[0].legend(fontsize=10)

# Subplot 2: Accuracy
ax[1].plot(epochs_range, sgd_res[5], 'r-', label='SGD - Train', linewidth=2)
ax[1].plot(epochs_range, sgd_res[6], 'r--', label='SGD - Test', linewidth=2)
ax[1].plot(epochs_range, adam_res[5], 'b-', label='Adam - Train', linewidth=2)
ax[1].plot(epochs_range, adam_res[6], 'b--', label='Adam - Test', linewidth=2)
ax[1].set_title("Évolution de l'Accuracy au fil des époques", fontsize=12)
ax[1].set_xlabel("Époque")
ax[1].set_ylabel("Accuracy")
ax[1].grid(True, alpha=0.3)
ax[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

### 🔲 Matrices de Confusion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Matrices de Confusion sur le Test Set (Chiffres 3 vs 8)", fontsize=14, fontweight='bold', y=1.05)

for i, (model_name, cm) in enumerate(matrices_confusion.items()):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Chiffre 3', 'Chiffre 8'])
    disp.plot(ax=axes[i], cmap='Blues', values_format='d')
    axes[i].set_title(model_name, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 5️⃣ Tableau Récapitulatif Final

In [ ]:
df_results = pd.DataFrame({
    'Méthode': ['SVM (RBF/SMO)', 'MLP - SGD (Momentum)', 'MLP - Adam'],
    'Type d\'optimisation': ['Convexe (SMO)', 'Non-convexe (SGD)', 'Non-convexe (Adam)'],
    'Temps d\'entraînement (s)': [results_time['SVM (RBF/SMO)'], results_time['MLP - SGD (Momentum)'], results_time['MLP - Adam']],
    'Accuracy (Test %)': [f"{results_acc['SVM (RBF/SMO)']*100:.2f}%", f"{results_acc['MLP - SGD (Momentum)']*100:.2f}%", f"{results_acc['MLP - Adam']*100:.2f}%"]
})

print("\n" + "="*70)
print("          📊 TABLEAU COMPARATIF FINAL")
print("="*70)
print(df_results.to_string(index=False))
print("="*70)

In [ ]:
# Graphique en barres pour la comparaison visuelle
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comparaison Globale des Méthodes", fontsize=14, fontweight='bold')

colors = ['#2ecc71', '#e74c3c', '#3498db']
methods = list(results_acc.keys())

# Barres Accuracy
bars1 = ax1.bar(methods, [v*100 for v in results_acc.values()], color=colors, edgecolor='black', linewidth=0.5)
ax1.set_title("Accuracy sur le Test Set (%)")
ax1.set_ylabel("Accuracy (%)")
ax1.set_ylim(90, 100)
for bar, val in zip(bars1, results_acc.values()):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2, f'{val*100:.2f}%', ha='center', fontweight='bold')

# Barres Temps
bars2 = ax2.bar(methods, results_time.values(), color=colors, edgecolor='black', linewidth=0.5)
ax2.set_title("Temps d'entraînement (secondes)")
ax2.set_ylabel("Temps (s)")
for bar, val in zip(bars2, results_time.values()):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1, f'{val:.2f}s', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 📝 Conclusion

### Résumé des observations :

| Critère | SVM (SMO) | MLP + SGD | MLP + Adam |
|---------|-----------|-----------|------------|
| Type d'optimisation | Convexe | Non-convexe | Non-convexe |
| Garantie d'optimalité | ✅ Minimum global | ❌ Minimum local | ❌ Minimum local |
| Vitesse de convergence | Variable | Lente | Rapide |
| Hyperparamètres | C, gamma | lr, momentum | lr, beta1, beta2 |

### Points clés :
1. **SVM** : Garantit le minimum global grâce à la convexité du problème dual
2. **SGD + Momentum** : Convergence plus lente mais bon contrôle du learning rate
3. **Adam** : Convergence rapide grâce à l'adaptation automatique du learning rate par paramètre